# Lesson 3.2 — Reading and Writing Data

**Objectives**
- Read CSV/Excel/JSON files into a DataFrame with `pd.read_*`
- Write a DataFrame back out to disk
- Inspect and fix common loading issues (encodings, delimiters, header rows)

See `modules/03-pandas/notes.md` (Lesson 3.2) for the full written explanation.

This notebook writes small practice files into `modules/03-pandas/notebooks/scratch/`
— that folder is exercise output only, not course material.


In [1]:
import pandas as pd
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
SCRATCH

PosixPath('scratch')

## `load_customers()` is really just `pd.read_csv`

In [2]:
from data_science_course.datasets import load_customers, load_products

customers_direct = pd.read_csv("../../../data/raw/customers.csv")
customers_via_loader = load_customers()
customers_direct.equals(customers_via_loader)

True

Both give identical results — but in this course we always use `load_customers()`
(never a hardcoded path), because it also regenerates the file deterministically if
it's ever missing.

## Dates load as plain text unless you say otherwise

In [3]:
print(load_customers()["signup_date"].dtype)  # object -- plain strings

object


In [4]:
customers_parsed = pd.read_csv("../../../data/raw/customers.csv", parse_dates=["signup_date"])
customers_parsed["signup_date"].dtype  # datetime64[ns] -- now real dates

dtype('<M8[ns]')

## Writing a DataFrame out, then reading it back

In [5]:
cleaned = customers_parsed.copy()
cleaned["region"] = cleaned["region"].str.strip().str.lower().str.title()
cleaned.to_csv(SCRATCH / "customers_cleaned.csv", index=False)

reloaded = pd.read_csv(SCRATCH / "customers_cleaned.csv")
reloaded.head()

,customer_id,signup_date,region,age,membership_tier,acquisition_channel,churned
0,C00001,2022-09-08,South,18.0,Silver,referral,1
1,C00002,2024-07-22,West,64.0,Bronze,email,0
2,C00003,2024-07-15,North,36.0,Gold,social,1
3,C00004,2023-06-16,South,22.0,Silver,paid_search,0
4,C00005,2024-08-17,West,23.0,Silver,organic,0


Note we passed `index=False` when writing. Try removing it and re-reading the file —
you'll see an extra unnamed index column show up. Forgetting `index=False` is one of
the most common beginner mistakes when writing CSVs.

## Excel: write, then read back

In [6]:
products = load_products()
products.head(20).to_excel(SCRATCH / "top_products.xlsx", index=False)

reloaded_products = pd.read_excel(SCRATCH / "top_products.xlsx")
reloaded_products.head()

,product_id,product_name,category,price,cost,launch_date
0,P0001,Mini Headphones,Electronics,409.02,142.95,2023-09-04
1,P0002,Classic Throw Pillow,Home & Kitchen,292.93,112.37,2024-01-06
2,P0003,Premium Shampoo,Beauty,38.78,20.17,2024-07-21
3,P0004,Pro Backpack,Sports & Outdoors,331.23,163.89,2021-11-15
4,P0005,Everyday Charger,Electronics,80.84,29.13,2023-06-04


In [7]:
reloaded_products.equals(products.head(20))

True

## JSON: write, then read back

In [8]:
products.head(5).to_json(SCRATCH / "products_sample.json", orient="records")

reloaded_json = pd.read_json(SCRATCH / "products_sample.json")
reloaded_json

,product_id,product_name,category,price,cost,launch_date
0,P0001,Mini Headphones,Electronics,409.02,142.95,2023-09-04
1,P0002,Classic Throw Pillow,Home & Kitchen,292.93,112.37,2024-01-06
2,P0003,Premium Shampoo,Beauty,38.78,20.17,2024-07-21
3,P0004,Pro Backpack,Sports & Outdoors,331.23,163.89,2021-11-15
4,P0005,Everyday Charger,Electronics,80.84,29.13,2023-06-04


`orient="records"` writes `[{"col": val, ...}, ...]` -- the most common, most portable
shape for tabular JSON.

## Try it yourself

1. Write `customers` out to `scratch/customers_no_index.csv` **without** `index=False`,
   read it back, and look at the extra column that appears. Then fix it.
2. Read `scratch/top_products.xlsx` back in with `pd.read_excel`, keeping only the
   `product_name` and `price` columns (hint: pass `usecols=`).
3. `orders.csv` doesn't have a messy delimiter or header offset issue, but pretend it
   might: read it with `pd.read_csv(..., sep=",")` explicitly and confirm the shape
   matches `load_orders().shape`.
4. Write the first 10 rows of `products` to `scratch/products_head.json` with
   `orient="records"`, then read it back and confirm `.shape` is `(10, 6)`.


In [9]:
# 1. TODO: write customers without index=False, read back, observe, then fix


# 2. TODO: read top_products.xlsx with usecols=


# 3. TODO: read orders.csv with an explicit sep, compare shape to load_orders()


# 4. TODO: write/read products_head.json, confirm shape


### Solution

In [10]:
from data_science_course.datasets import load_orders

# 1.
customers = load_customers()
customers.to_csv(SCRATCH / "customers_no_index.csv")  # missing index=False on purpose
bad_reload = pd.read_csv(SCRATCH / "customers_no_index.csv")
print(bad_reload.columns[0])  # "Unnamed: 0" -- the accidental index column

customers.to_csv(SCRATCH / "customers_no_index.csv", index=False)  # fixed
good_reload = pd.read_csv(SCRATCH / "customers_no_index.csv")
print(good_reload.columns[0])  # "customer_id" -- as intended

# 2.
name_and_price = pd.read_excel(SCRATCH / "top_products.xlsx", usecols=["product_name", "price"])
print(name_and_price.head())

# 3.
orders_explicit = pd.read_csv("../../../data/raw/orders.csv", sep=",")
print(orders_explicit.shape == load_orders().shape)

# 4.
products.head(10).to_json(SCRATCH / "products_head.json", orient="records")
products_head_reloaded = pd.read_json(SCRATCH / "products_head.json")
print(products_head_reloaded.shape)

Unnamed: 0
customer_id
           product_name   price
0       Mini Headphones  409.02
1  Classic Throw Pillow  292.93
2       Premium Shampoo   38.78
3          Pro Backpack  331.23
4      Everyday Charger   80.84
True
(10, 6)
